### ODEV

 Gerekli K?t?phanelerin ?ndirilmesi
Jupyter Notebook dosyam?z?n ba??na a?a??daki gibi gerekli k?t?phaneleri import ederek ba?layabiliriz:

In [ ]:
# Gerekli k?t?phanelerin y?klenmesi
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")


DataSheet_1_Nomogram for predicting invasive lung adenocarcinoma in small solitary pulmonary nodules veri setini y?klemek i?in bir CSV dosyas? olarak indirin veya veri setini do?rudan y?klemek i?in pandas k?t?phanesini kullan?n.

In [ ]:
# Veri setinin y?klenmesi
data = pd.read_csv('DataSheet_1_Nomogram for predicting invasive lung adenocarcinoma in small solitary pulmonary nodules.csv')

In [ ]:
print(data)

In [ ]:
# ?lk birka? sat?r? g?r?nt?leme
data.head()

Veri setini inceleyerek, s?tun isimlerini ve veri t?rlerini g?zden ge?erek. Eksik veri olup olmad???n? kontrol edelim.

In [ ]:
# S?tunlar ve veri t?rleri
print(data.info())



In [ ]:
# Eksik verileri kontrol etme
print(data.isnull().sum())


E?er eksik veri varsa, ilgili s?tunlar? doldurma veya kald?rma i?lemi yap?lmal?d?r. Ancak bu veri setinde eksik veri olmad???n? varsayal?m.

Ba??ms?z de?i?kenleri (?zellikler) ve ba??ml? de?i?keni (hedef) belirleyelim. Bu veri setinde, Tumor_Invasiveness s?tunu hedef de?i?kendir.

In [ ]:
# ?zellikler ve hedef de?i?kenin ayr?lmas?
X = data.drop("Pleural_Adhesions", axis=1)


In [ ]:
print(X)

In [ ]:
y = data["Tumor_Invasiveness"]

In [ ]:
print(y)

E?itim ve test setlerini %80 e?itim, %20 test olacak ?ekilde ay?ral?m.

In [ ]:
# E?itim ve test setlerine ay?rma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
print(X_train, X_test, y_train, y_test)

Her bir modeli e?itim verisi ?zerinde e?itece?iz ve test verisi ?zerinde performanslar?n? de?erlendirece?iz.

 ?oklu Lineer Regresyon (MLR)

In [ ]:
# ?oklu Lineer Regresyon Modeli
mlr = LinearRegression()
mlr.fit(X_train, y_train)
y_pred_mlr = mlr.predict(X_test)


In [ ]:
y_pred_mlr

Polinom Regresyon (3. derece)

In [ ]:
# Polinom ?zellikler (3. derece)
poly_features = PolynomialFeatures(degree=3)
X_poly_train = poly_features.fit_transform(X_train)
X_poly_test = poly_features.transform(X_test)

# Polinom Regresyon Modeli
plr = LinearRegression()
plr.fit(X_poly_train, y_train)
y_pred_plr = plr.predict(X_poly_test)


In [ ]:
y_pred_plr

Karar A?ac? (DT)

In [ ]:
# Karar A?ac? Modeli
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)


In [ ]:
y_pred_dt

 Rastgele Orman (RF)

In [ ]:
# Rastgele Orman Modeli
rf = RandomForestRegressor(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


In [ ]:
y_pred_rf

Her bir model i?in r2, mae, mape, mse skorlar?n? hesaplayaca??z.

In [ ]:
# Skor hesaplama fonksiyonu
def evaluate_model(y_test, y_pred):
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    return r2, mae, mse, mape

# Model de?erlendirmeleri
results = {
    "MLR": evaluate_model(y_test, y_pred_mlr),
    "PLR (3rd Degree)": evaluate_model(y_test, y_pred_plr),
    "DT": evaluate_model(y_test, y_pred_dt),
    "RF": evaluate_model(y_test, y_pred_rf)
}

# Sonu?lar? DataFrame olarak g?r?nt?leme
results_df = pd.DataFrame(results, index=["R2", "MAE", "MSE", "MAPE"])
print(results_df)


Her bir modelin performans?n? analiz edin. R2 de?eri 1'e ne kadar yak?nsa model o kadar ba?ar?l?d?r. MAE, MSE ve MAPE ise hatalar?n boyutunu g?sterir, d???k de?erler daha iyi performans anlam?na gelir.